<a href="https://colab.research.google.com/github/navacron/aistuff/blob/main/PartialInference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers peft accelerate

import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model


In [ ]:
base_model_name = "gpt2"   # or "tiiuae/falcon-7b-instruct" on a bigger GPU

config = AutoConfig.from_pretrained(base_model_name)
config.output_hidden_states = True  # crucial for grabbing layer-H activations

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    config=config,
    torch_dtype=torch.float16,
    device_map="auto"
)
base_model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

3. Run once up to layer H and cache the activations

Here we treat layer index H in the Transformer block stack. For GPT-2, config.n_layer is the number of blocks.

In [ ]:
H = 6  # example: cache after block 6 (0-based index)

@torch.no_grad()
def forward_to_layer_H(input_ids, attention_mask=None, H=H):
    outputs = base_model.transformer(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        use_cache=False,
    )
    hidden_states = outputs.hidden_states
    hidden_H = hidden_states[H + 1]  # 0 = embeddings
    return hidden_H


You’d typically wrap this in a cache keyed by your input text / IDs:

In [ ]:
hidden_cache = {}

def get_cached_hidden_H(text, H=H):
    if text in hidden_cache:
        return hidden_cache[text]

    enc = tokenizer(text, return_tensors="pt").to(base_model.device)
    hidden_H = forward_to_layer_H(enc["input_ids"], enc["attention_mask"], H=H)
    hidden_cache[text] = (hidden_H, enc["attention_mask"])
    return hidden_cache[text]


4. Build a “tail model” (layers H..end) with LoRA

Idea:

We reuse the base model’s first H blocks by calling forward_to_layer_H.

Then build a module that only runs blocks H..end + final LN + LM head.

We apply LoRA only to these tail blocks, so we can have multiple adapters there.

In [ ]:
import torch.nn as nn

class GPT2Tail(nn.Module):
    def __init__(self, base_model, H):
        super().__init__()
        self.H = H
        self.config = base_model.config  # PEFT may inspect this

        self.tail_blocks = nn.ModuleList(base_model.transformer.h[H:])
        self.ln_f = base_model.transformer.ln_f
        self.lm_head = base_model.lm_head

    def forward(self, hidden_states, attention_mask=None):
        x = hidden_states
        for block in self.tail_blocks:
            x = block(x, attention_mask=attention_mask)[0]
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits

    # just to keep PEFT from complaining; we won't call .generate()
    def prepare_inputs_for_generation(self, *args, **kwargs):
        raise NotImplementedError(
            "GPT2Tail does not support .generate(); "
            "call it directly with hidden states."
        )



Now wrap that tail with LoRA via PEFT:

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["c_attn", "c_proj"],
    bias="none",
    task_type="CAUSAL_LM",
    fan_in_fan_out=True,
)

tail_adapterA = get_peft_model(GPT2Tail(base_model, H=H), lora_config).to(base_model.device)
tail_adapterB = get_peft_model(GPT2Tail(base_model, H=H), lora_config).to(base_model.device)

tail_adapterA.print_trainable_parameters()
tail_adapterB.print_trainable_parameters()



trainable params: 405,504 || all params: 81,531,648 || trainable%: 0.4974
trainable params: 405,504 || all params: 81,531,648 || trainable%: 0.4974


5. Using multiple LoRA tails on the same cached activations

You can create multiple LoRA-augmented tails that all share the same underlying weights but have different adapters:

In [ ]:
# Adapter A
#tail_adapterA = get_peft_model(GPT2Tail(base_model, H=H), lora_config).to(base_model.device)
# ... fine-tune tail_adapterA on task A, then save

# Adapter B
#tail_adapterB = get_peft_model(GPT2Tail(base_model, H=H), lora_config).to(base_model.device)
# ... fine-tune tail_adapterB on task B, then save


Infernece Time

In [ ]:
@torch.no_grad()
def run_with_adapter(tail_model, text):
    hidden_H, attention_mask = get_cached_hidden_H(text)
    logits = tail_model(hidden_H, attention_mask=attention_mask)
    # standard causal LM decoding: take last token, softmax, etc.
    last_logits = logits[:, -1, :]
    probs = last_logits.softmax(dim=-1)
    topk = torch.topk(probs, 5)
    tokens = [tokenizer.decode([i]) for i in topk.indices[0].tolist()]
    return list(zip(tokens, topk.values[0].tolist()))

print("Adapter A:")
print(run_with_adapter(tail_adapterA, "The capital of France is"))

print("Adapter B:")
print(run_with_adapter(tail_adapterB, "The capital of France is"))


Adapter A:


TypeError: GPT2Tail.forward() got an unexpected keyword argument 'input_ids'